In [1]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
import pandas

device = "mps"
print(torch.backends.mps.is_available())

True


In [2]:
ds = load_dataset("cais/mmlu", "high_school_physics", split="test").shuffle(seed=100).select(range(50))
row = ds[0]
print(row["question"])
print(row["choices"])
print(row["answer"])

A nonconducting sphere is given a nonzero net electric charge, +Q, and then brought close to a neutral conducting sphere of the same radius. Which of the following will be true?
['An electric field will be induced within the conducting sphere.', 'The conducting sphere will develop a net electric charge of –Q.', 'The spheres will experience an electrostatic attraction.', 'The spheres will experience an electrostatic repulsion.']
2


In [4]:
model_name = "allenai/OLMo-2-0425-1B"
tok = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, dtype=torch.bfloat16).to(device)
model.eval()

Loading weights:   0%|          | 0/179 [00:00<?, ?it/s]

Olmo2ForCausalLM(
  (model): Olmo2Model(
    (embed_tokens): Embedding(100352, 2048, padding_idx=100277)
    (layers): ModuleList(
      (0-15): 16 x Olmo2DecoderLayer(
        (self_attn): Olmo2Attention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (v_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (q_norm): Olmo2RMSNorm((2048,), eps=1e-06)
          (k_norm): Olmo2RMSNorm((2048,), eps=1e-06)
        )
        (mlp): Olmo2MLP(
          (gate_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (up_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (down_proj): Linear(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (post_attention_layernorm): Olmo2RMSNorm((2048,), eps=1e-06

In [ ]:
letters = ["A", "B", "C", "D"]
letter_ids = []
for letter in letters:
    ids = tok.encode(" " + letter, add_special_tokens=False)
    assert len(ids) == 1
    letter_ids.append(ids[0])

def format_prompt(row):
    text = row["question"] + "\n"
    for letter, choice in zip(letters, row["choices"]):
        text += f"{letter}. {choice}\n"
    text += "Answer:"
    return text

prompt = format_prompt(row)
print(prompt)

A nonconducting sphere is given a nonzero net electric charge, +Q, and then brought close to a neutral conducting sphere of the same radius. Which of the following will be true?
A. An electric field will be induced within the conducting sphere.
B. The conducting sphere will develop a net electric charge of –Q.
C. The spheres will experience an electrostatic attraction.
D. The spheres will experience an electrostatic repulsion.
Answer:


In [6]:
def get_choice_probs(prompt, letter_ids):


    inputs = tok(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        out = model(input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"])

    next_token_scores = out.logits[0, -1]
    letter_scores = next_token_scores[letter_ids].float()
    probs = torch.softmax(letter_scores, dim=-1)
    all_probs = torch.softmax(next_token_scores.float(), dim=-1)
    letter_mass = all_probs[letter_ids].sum().item()
    return probs.tolist(), letter_mass
